# ECHO MOVE — Backend trên Colab
Chạy lần lượt các cell. Cell 4 in ra **link web** đã gắn sẵn địa chỉ server; mở link đó là dùng được.

> Nên chọn Runtime → GPU (T4) để YOLO nhanh hơn. Giữ tab Colab mở trong lúc demo.

In [ ]:
# 1. Lấy mã nguồn
REPO = 'https://github.com/FSB-MSA36HN/echo-move.git'
PAGES = 'https://fsb-msa36hn.github.io/echo-move/'   # URL GitHub Pages của web
!rm -rf echo-move && git clone -q $REPO
%cd echo-move

In [ ]:
# 2. Cài backend + YOLO + MediaPipe (1-3 phút)
!pip install -q "./backend[ai]"

In [ ]:
# 3. Cài cloudflared (tunnel miễn phí, không cần tài khoản)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
# 4. Chạy server + tunnel, in link web
import subprocess, re, time, os, threading
env = dict(os.environ, ECHO_PORT='8000')
server = subprocess.Popen(['python', '-m', 'echo_move'], cwd='backend', env=env)
time.sleep(8)
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                          stderr=subprocess.PIPE, text=True)
url = None
for line in tunnel.stderr:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('Server :', url)
print('Health :', url + '/health')
print('Mở web :', f'{PAGES}?server={url}')
threading.Thread(target=lambda: [None for _ in tunnel.stderr], daemon=True).start()
server.wait()   # giữ cell chạy; Stop để tắt